# 06 · Assumption checks (Statistical Analysis; Supplementary Table S6)

Normality of each measure within each group (Shapiro–Wilk, skewness, excess kurtosis) and homogeneity of variance
across groups (Levene and Brown–Forsythe tests, computed in notebook 04). Run notebook 04 first.

In [ ]:
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import pandas as pd
from scipy import stats
import dbiat_analysis as A

cfg = A.load_settings()
df = pd.read_csv(A.path(cfg, "derived", "scores.csv"))
anova = pd.read_csv(A.path(cfg, "tables", "anova.csv"))

## 1. Normality

In [ ]:
rows = []
for m in A.MEASURES:
    for g in A.GROUPS:
        x = df.loc[df.group == g, m].dropna()
        W, p = stats.shapiro(x)
        rows.append(dict(measure=m, group=g, n=len(x), shapiro_W=W, shapiro_p=p,
                         skewness=stats.skew(x), excess_kurtosis=stats.kurtosis(x)))
norm = pd.DataFrame(rows)
norm["non_normal_p<.05"] = norm.shapiro_p < .05
A.write_table(norm, cfg, "assumption_normality")
print("Measure × group cells with Shapiro–Wilk p < .05:", int(norm["non_normal_p<.05"].sum()), "of", len(norm))
norm.pivot(index="measure", columns="group", values="shapiro_p").reindex(A.MEASURES).round(4)

## 2. Homogeneity of variance

In [ ]:
var = anova[["measure", "levene_W", "levene_p", "brown_forsythe_W", "brown_forsythe_p"]]
A.write_table(var, cfg, "assumption_variance")
var.round(4)